# 04 · Silver: clean, typed, de-duplicated, PII-aware
| Bronze problem | Silver fix |
|---|---|
| dates as text (`15-Oct-2026`) | real `DATE` columns |
| amounts as text | `DECIMAL(18,2)` |
| suppliers as arrays (joint ventures) | separate table `contract_suppliers`, 1 row per supplier |
| messy supplier names | normalised `supplier_key` |
| personal data (contact names, emails, phones) | dropped (data minimisation) or tagged `pii` |
| possible duplicates across runs | keep the latest version of each id |
| **hidden join key** in reference numbers | `step_activity_id` extracted, links notices to contracts |

In [0]:
-- =====================================================================
-- CELL 1: SILVER CONTRACTS -> typed, de-duplicated, with a join key
-- =====================================================================
-- try_to_timestamp(text, format) -> converts text to a timestamp.
--   'try_' = returns NULL instead of crashing if a value is badly formatted.
--   'dd-MMM-yyyy' = 2-digit day, 3-letter month, 4-digit year (15-Oct-2026)
-- try_cast(x AS DECIMAL(18,2)) -> text to number with 2 decimals (money). NULL if invalid.
-- regexp_extract(...) -> pulls the STEP activity number out of the reference:
--   'KE-MOTI-545201-CS-CDS' -> '545201'
--   pattern: a dash, 5 to 7 digits (captured), a dash, 2 capital letters, a dash
-- QUALIFY ROW_NUMBER() ... = 1 -> DE-DUPLICATION: if the same contract appears in
--   several runs, keep only the newest (latest run_date, latest ingestion).
-- CLUSTER BY (project_id) -> LIQUID CLUSTERING: Databricks stores rows of the same
--   project close together, so "all contracts of project P123" reads fewer files.
-- teammemfullname (a World Bank staff name) is deliberately NOT selected:
--   DATA MINIMISATION -> we don't copy personal data we don't need.

CREATE OR REPLACE TABLE procintel.silver.contracts
CLUSTER BY (project_id)
COMMENT 'Clean contract awards (Kenya). One row per contract. Source: bronze.contracts (STEP via contractdata API)'
AS
SELECT
  CAST(contr_id AS STRING)                                  AS contract_id,
  projectid                                                 AS project_id,
  project_name,
  contr_desc                                                AS contract_description,
  contr_refnum                                              AS contract_reference,
  regexp_extract(contr_refnum, '-([0-9]{5,7})-[A-Z]{2}-', 1) AS step_activity_id,
  procurement_group                                         AS procurement_group_code, -- CW/GO/CS/NC
  procurement_group_desc                                    AS procurement_group,
  procu_meth_text                                           AS procurement_method,
  rvw_type                                                  AS review_type,            -- Prior / Post
  CAST(try_to_timestamp(contr_sgn_date,   'dd-MMM-yyyy') AS DATE) AS signing_date,
  CAST(try_to_timestamp(contr_no_obj_dat, 'dd-MMM-yyyy') AS DATE) AS no_objection_date,
  try_cast(total_contr_amnt AS DECIMAL(18,2))               AS total_amount_usd,
  size(supp_name)                                           AS n_suppliers,            -- >1 = joint venture
  concat_ws('; ', mjsecname)                                AS major_sector,   -- array -> plain text, ONE format everywhere
  regionname                                                AS region,
  countryshortname                                          AS country,
  _run_date, _ingested_at                                   -- lineage carried forward
FROM procintel.bronze.contracts
QUALIFY ROW_NUMBER() OVER (PARTITION BY contr_id ORDER BY _run_date DESC, _ingested_at DESC) = 1;

In [0]:
-- =====================================================================
-- CELL 2: SILVER CONTRACT_SUPPLIERS -> explode the supplier arrays
-- =====================================================================
-- A joint-venture contract has ["FIRM A","FIRM B"] and ["100","200"].
-- posexplode(array) -> one row per element, plus its POSITION (0, 1, 2...).
-- element_at(array, pos+1) -> the amount at the SAME position
--   (element_at counts from 1, posexplode counts from 0, hence +1).
-- supplier_key = a NORMALISED name, so spelling variants group together:
--   1. upper case        2. remove "P.O. BOX ..." and everything after it
--   3. remove punctuation 4. remove legal suffixes (LTD, LIMITED, CO, COMPANY, PLC, INC)
--   5. squeeze repeated spaces
-- This is ENTITY RESOLUTION (simple, rule-based). Upgrade path: fuzzy matching.

CREATE OR REPLACE TABLE procintel.silver.contract_suppliers
COMMENT 'One row per (contract, supplier). Joint ventures are split. supplier_name may be an individual -> tagged PII.'
AS
WITH exploded AS (
  SELECT CAST(contr_id AS STRING) AS contract_id,
         projectid                AS project_id,
         pos,
         supplier_name,
         try_cast(element_at(supplier_contr_amount, pos + 1) AS DECIMAL(18,2)) AS supplier_amount_usd,
         _run_date, _ingested_at
  FROM procintel.bronze.contracts
  LATERAL VIEW posexplode(supp_name) AS pos, supplier_name
  QUALIFY ROW_NUMBER() OVER (PARTITION BY contr_id, pos ORDER BY _run_date DESC, _ingested_at DESC) = 1
)
SELECT *,
  trim(regexp_replace(
     regexp_replace(
       regexp_replace(
         regexp_replace(upper(supplier_name), 'P\\.? ?O\\.? ?BOX.*$', ''),     -- drop address
       '[^A-Z0-9 ]', ' '),                                                    -- drop punctuation
     '\\b(LTD|LIMITED|CO|COMPANY|PLC|INC|CORP|CORPORATION)\\b', ''),           -- drop legal suffixes
  ' +', ' '))                                                                 AS supplier_key
FROM exploded;

In [0]:
-- =====================================================================
-- CELL 3: SILVER NOTICES -> typed dates, join key, NO personal contact data
-- =====================================================================
-- Two date formats in one source (real-world messiness):
--   noticedate               '03-Nov-2026'            -> needs the format pattern
--   submission_deadline_date '2026-12-03T00:00:00Z'   -> ISO 8601, try_cast understands it
-- DATA MINIMISATION: contact_name, contact_email, contact_phone_no are NOT copied.
--   The agency (contact_organization) is kept, because it's an organisation, not a person.

CREATE OR REPLACE TABLE procintel.silver.notices
CLUSTER BY (project_id)
COMMENT 'Clean procurement notices (Kenya). Personal contact fields removed (data minimisation).'
AS
SELECT
  id                                                         AS notice_id,
  project_id,
  project_name,
  notice_type,
  CAST(try_to_timestamp(noticedate, 'dd-MMM-yyyy') AS DATE)  AS notice_date,
  CAST(try_cast(submission_deadline_date AS TIMESTAMP) AS DATE) AS submission_deadline,
  procurement_method_name                                    AS procurement_method,
  procurement_group                                          AS procurement_group_code,
  bid_reference_no,
  regexp_extract(bid_reference_no, '-([0-9]{5,7})-[A-Z]{2}-', 1) AS step_activity_id,
  bid_description,
  notice_text,                                               -- semi-structured text: features + RAG later
  contact_organization                                       AS implementing_agency,
  notice_status,
  _run_date, _ingested_at
FROM procintel.bronze.notices
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY _run_date DESC, _ingested_at DESC) = 1;

In [0]:
-- =====================================================================
-- CELL 4: SILVER INDICATORS -> flatten the nested JSON
-- =====================================================================
-- The Indicators API nests things: indicator = {id, value}, country = {id, value}.
-- "indicator.id" reads the field 'id' INSIDE the struct 'indicator'.
-- Rows with value = null (years with no score) are dropped.

CREATE OR REPLACE TABLE procintel.silver.governance_indicators
COMMENT 'Kenya governance scores per year: CPIA (1-6) and WGI 2025 revision (-2.5..+2.5)'
AS
SELECT
  indicator.id         AS indicator_code,
  indicator.value      AS indicator_name,
  countryiso3code      AS country_iso3,
  CAST(date AS INT)    AS year,
  CAST(value AS DOUBLE) AS score,
  _run_date, _ingested_at
FROM procintel.bronze.indicators
WHERE value IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY indicator.id, date ORDER BY _run_date DESC, _ingested_at DESC) = 1;

In [0]:
-- =====================================================================
-- CELL 5: COLUMN TAGS -> mark personal data so governance tools can find it
-- =====================================================================
-- supplier_name can be a PERSON (individual consultants, e.g. "DORIS NJAMBI MWANGI").
-- Tagging it pii = 'possible' lets us (a) search the catalog for all PII columns,
-- (b) attach masking policies later, (c) prove to auditors we know where PII lives.

ALTER TABLE procintel.silver.contract_suppliers ALTER COLUMN supplier_name SET TAGS ('pii' = 'possible');
ALTER TABLE procintel.silver.contracts SET TAGS ('layer' = 'silver', 'grain' = 'one_row_per_contract');
ALTER TABLE procintel.silver.notices   SET TAGS ('layer' = 'silver', 'grain' = 'one_row_per_notice');

In [0]:
-- =====================================================================
-- CELL 6: DATA QUALITY + does the hidden join key actually work?
-- =====================================================================
-- Part 1: how many dates/amounts failed to parse? (NULL after try_ functions)
-- Part 2: link each contract to the FIRST notice of the same STEP activity
--         (excluding 'Contract Award' notices, which come AFTER signing)
--         -> tender_days = signing_date - first notice date

WITH first_notice AS (
  SELECT step_activity_id, MIN(notice_date) AS first_notice_date
  FROM procintel.silver.notices
  WHERE step_activity_id <> '' AND notice_type <> 'Contract Award'
  GROUP BY step_activity_id
)
SELECT
  COUNT(*)                                                        AS contracts,
  COUNT_IF(signing_date IS NULL)                                  AS bad_signing_dates,
  COUNT_IF(total_amount_usd IS NULL)                              AS bad_amounts,
  COUNT_IF(c.step_activity_id <> '')                              AS with_activity_id,
  COUNT(n.first_notice_date)                                      AS linked_to_a_notice,
  ROUND(100 * COUNT(n.first_notice_date) / COUNT(*), 1)           AS pct_linked,
  median(datediff(c.signing_date, n.first_notice_date))           AS median_tender_days,
  COUNT_IF(datediff(c.signing_date, n.first_notice_date) < 0)     AS negative_durations   -- should be ~0
FROM procintel.silver.contracts c
LEFT JOIN first_notice n
  ON c.step_activity_id = n.step_activity_id AND c.step_activity_id <> '';

In [0]:
-- Which columns does bronze.projects actually have?
SELECT concat_ws(', ', collect_list(column_name)) AS all_columns
FROM procintel.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'projects';

In [0]:
-- =====================================================================
-- CELL 7: SILVER PROJECTS -> one row per World Bank project in Kenya
-- =====================================================================
-- coalesce(a, b) = take the first value that isn't NULL.
-- We use it to try TWO date formats, because this API mixes formats:
--   '2023-06-28T00:00:00Z'      (ISO)          -> try_cast(... AS TIMESTAMP)
--   '6/30/2028 12:00:00 AM'     (US style)     -> 'M/d/yyyy h:mm:ss a'
-- NOTE: 'pdo' (development objective) is NOT in the API's default field set.
--   We verified the real columns via information_schema before fixing this.
-- DATA MINIMISATION: 'teamleadname' (a staff member's name) is deliberately NOT copied.

CREATE OR REPLACE TABLE procintel.silver.projects
COMMENT 'Clean World Bank projects (Kenya). One row per project.'
AS
SELECT
  id                                       AS project_id,
  project_name,
  status,                                  -- Active / Closed / Pipeline / Dropped
  impagency                                AS implementing_agency,
  borrower,
  envassesmentcategorycode                 AS env_category,   -- A (highest risk) / B / C
  lendinginstr                             AS lending_instrument,
  CAST(coalesce(try_cast(boardapprovaldate AS TIMESTAMP),
                try_to_timestamp(boardapprovaldate, 'M/d/yyyy h:mm:ss a')) AS DATE) AS approval_date,
  CAST(coalesce(try_cast(closingdate AS TIMESTAMP),
                try_to_timestamp(closingdate, 'M/d/yyyy h:mm:ss a')) AS DATE)       AS closing_date,
  try_cast(regexp_replace(CAST(curr_total_commitment AS STRING), ',', '') AS DOUBLE) * 1e6 AS total_commitment_usd,
  _run_date, _ingested_at
FROM procintel.bronze.projects
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY _run_date DESC, _ingested_at DESC) = 1;

In [0]:
-- =====================================================================
-- CELL 8: SILVER DOCUMENTS -> document catalog with a "doc_family" and AUTHORITY
-- =====================================================================
-- doc_family groups the many document types into the few we use.
-- authority = how much we trust each type when sources DISAGREE (Module 3):
--   IEG reviews (independent evaluators)       0.90
--   ICR (written at project closing)            0.75
--   PAD / appraisal (written at the start)      0.60
--   ISR (progress reports DURING the project)   0.50  <- often optimistic
-- A document can list several projects ("P1,P2"); we take the first as primary.

CREATE OR REPLACE TABLE procintel.silver.documents
COMMENT 'Document catalog (Kenya) with doc_family and authority weight. Full text is loaded separately.'
AS
SELECT
  CAST(id AS STRING)                                    AS doc_id,
  trim(element_at(split(projectid, '[,; ]+'), 1))       AS project_id,
  docty                                                 AS doc_type,
  CASE
    WHEN docty = 'Implementation Completion Report Review'      THEN 'IEG_ICRR'
    WHEN docty ILIKE 'Project Performance Assessment Report%'   THEN 'IEG_PPAR'
    WHEN docty = 'Implementation Completion and Results Report' THEN 'ICR'
    WHEN docty = 'Implementation Status and Results Report'     THEN 'ISR'
    WHEN docty IN ('Project Appraisal Document','Project Paper','Staff Appraisal Report') THEN 'PAD'
    WHEN docty = 'Procurement Plan'                              THEN 'PROC_PLAN'
    ELSE 'OTHER'
  END                                                   AS doc_family,
  CASE
    WHEN docty IN ('Implementation Completion Report Review') OR docty ILIKE 'Project Performance Assessment Report%' THEN 0.90
    WHEN docty = 'Implementation Completion and Results Report' THEN 0.75
    WHEN docty IN ('Project Appraisal Document','Project Paper','Staff Appraisal Report') THEN 0.60
    WHEN docty = 'Implementation Status and Results Report' THEN 0.50
    ELSE 0.40
  END                                                   AS authority,
  display_title                                         AS title,
  CAST(coalesce(try_cast(docdt AS TIMESTAMP),
                try_to_timestamp(docdt, 'M/d/yyyy h:mm:ss a')) AS DATE) AS doc_date,
  txturl, pdfurl, url,
  _run_date, _ingested_at
FROM procintel.bronze.documents
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY _run_date DESC, _ingested_at DESC) = 1;

In [0]:
-- =====================================================================
-- CELL 9: SANITY CHECK projects + documents
-- =====================================================================
SELECT 'projects' AS t, COUNT(*) AS n,
       COUNT_IF(approval_date IS NULL) AS null_dates,
       ROUND(percentile(total_commitment_usd, 0.5)/1e6, 1) AS median_commitment_musd,
       NULL AS docs_with_text
FROM procintel.silver.projects
UNION ALL
SELECT 'documents', COUNT(*), COUNT_IF(doc_date IS NULL), NULL, COUNT_IF(txturl IS NOT NULL)
FROM procintel.silver.documents;